# Chapter 15 — Your First Extension

**Companion to *Pi Agents*** · [`Pi Agents` chapter 15](https://programmer.ie/books/pi/15-chapter/)

| | |
|---|---|
| Chapter | `15-chapter.md` · weight 15 · `/books/pi/15-chapter/` |
| Notebook | `notebooks/pi/15-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Does registering a `direct` tool make it **active** — and does a call with no
arguments pass Pi's own schema validation?

The chapter's example tool has an optional `base` parameter. The first version
declared it as required while its description said optional, so a model following
the description sent arguments that did not match the schema. The fix and the test
are both about that boundary.


## What this notebook establishes

- Registering a `direct`-exposure tool makes it **active** immediately: it appears in `getActiveToolNames()` at registration time, not only after something calls `setActiveTools()`.
- A call with **no arguments** is valid when the parameter is `Type.Optional` and produces `content` plus `details`.
- A command registered with `pi.registerCommand` is listed beside the built-ins and runs without the model being asked.

## What this notebook does **not** establish

- **The example tool returns an instruction to the model rather than executing a command.** Execution is chapter 17's subject.
- **`registerToolRenderer` arrived in 1.0.1** and was not exercised in a running session by the chapter's suite (`ch17-lim3`).
- **No measurement** of how often a model calls a tool it was offered.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — the chapter's two extensions loaded unmodified into real sessions.
* **Evidence scope:** `in_process_runtime`. Registration, activation and schema validation are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(15))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/15-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: registered, active, and callable

Three sets again, and the difference between them is the chapter's first claim.
`review_scope` is a `direct`-exposure tool with an optional `base` parameter;
`hello` is a command.

In [3]:
EXT = pinb.driver_source("ch15-first-extension.ts")
ext = pinb.driver(EXT, label="ch15-first-extension", timeout=300)
inv = ext["inventory"]
print(pinb.md_table(
    ["set", "contains review_scope?", "contains hello?"],
    [["registered (getAllTools)", "yes" if "review_scope" in inv["registered"] else "no", "-"],
     ["active (getActiveToolNames)", "yes" if "review_scope" in inv["active"] else "no", "-"],
     ["callable (getCallableToolNames)", "yes" if "review_scope" in inv["callable"] else "no", "-"],
     ["commands", "-", "yes" if "hello" in inv["commands"] else "no"]],
))

| set | contains review_scope? | contains hello? |
|---|---|---|
| registered (getAllTools) | yes | - |
| active (getActiveToolNames) | yes | - |
| callable (getCallableToolNames) | yes | - |
| commands | - | yes |


In [4]:
pinb.show_checks([
    pinb.check("registering a direct tool makes it active at registration time",
               "review_scope" in inv["active"], "active immediately"),
    pinb.check("it is also registered and callable",
               "review_scope" in inv["registered"] and "review_scope" in inv["callable"],
               "both"),
    pinb.check("the command is listed beside the built-ins",
               "hello" in inv["commands"], "listed"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------------------  ------------------
PASS  registering a direct tool makes it active at registration time  active immediately
PASS  it is also registered and callable  both
PASS  the command is listed beside the built-ins  listed

3/3 assertions held.


## Experiment: a call with no arguments

The chapter's second test is the one that matters. `base` is optional, so a model
that sends `{}` — nothing — must get a valid result, not a schema error.

The driver scripts exactly that: a tool call with no arguments, then an answer.

In [5]:
print("A scripted call with no arguments:")
print(pinb.md_table(
    ["", "value"],
    [["isError", ext["noArgs"]["isError"]],
     ["content", ext["noArgs"]["content"]],
     ["details", ext["noArgs"]["details"]]],
))

pinb.show_checks([
    pinb.check("a no-argument call is valid", ext["noArgs"]["isError"] is False, "isError: false"),
    pinb.check("and returns content", bool(ext["noArgs"]["content"]), ext["noArgs"]["content"][:60]),
    pinb.check("and details", ext["noArgs"]["details"] is not None, str(ext["noArgs"]["details"])),
])

A scripted call with no arguments:
|  | value |
|---|---|
| isError | False |
| content | Run `git status --short` and report the changed files. |
| details | {'base': None, 'diffCmd': 'git status --short'} |
== Assertions ==
  assertion  observed
----  ---------------------------  ------------------------------------------------------
PASS  a no-argument call is valid  isError: false
PASS  and returns content  Run `git status --short` and report the changed files.
PASS  and details  {'base': None, 'diffCmd': 'git status --short'}

3/3 assertions held.


## Contrasting case: the schema the description promised

The chapter records the defect its own tests found. `base` was declared
`Type.String(...)` while its description said *Optional*. A model following the
description would send `{}` — and `{}` does not match a required string.

The fix is one word: `Type.Optional(Type.String(...))`. The test above is the
regression guard.

In [6]:
print("The chapter's own words:")
print("  \"base is optional: a call with no arguments is valid and defaults to the")
print("   working tree\"")
print()
print("The declaration that makes it true:")
print("  parameters: Type.Object({ base: Type.Optional(Type.String({...})) })")
print()
print("The declaration that broke it:")
print("  parameters: Type.Object({ base: Type.String({...}) })")
print()
assert ext["noArgs"]["isError"] is False
print("held: the schema matches the description, so a no-argument call is valid")

The chapter's own words:
  "base is optional: a call with no arguments is valid and defaults to the
   working tree"

The declaration that makes it true:
  parameters: Type.Object({ base: Type.Optional(Type.String({...})) })

The declaration that broke it:
  parameters: Type.Object({ base: Type.String({...}) })

held: the schema matches the description, so a no-argument call is valid


## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [7]:
PATTERNS = 'ch15-first-extension/first-extension.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch15-first-extension/first-extension.test.ts', show="tool")

$ node --test ch15-first-extension/first-extension.test.ts
  PASS  3 passed, 0 failed, 0 skipped  in 944 ms

  tests matching 'tool':
    ok   the tool is registered, visible to the model, and runs through the real pipeline


## Your turn: predict, then run

**Predict first.** Change `base` back to `Type.String(...)` (required). What does the
no-argument call return — and what does the model see?

**Then change one input.** `BASE` below is what the model sends. Try `{}` (nothing),
then `{"base": "main"}`, then `{"base": 42}` (wrong type).

**Predict the boundary.** What happens if the model sends an argument the schema
does not declare at all — `{"base": "main", "extra": true}`? Is that a schema error
or is the extra field ignored?

In [8]:
BASE = {}
print("predicted: with base optional, {} is valid and defaults to the working tree")
print()
print("observed above:")
print("  isError:", ext["noArgs"]["isError"])
print("  content:", ext["noArgs"]["content"][:60])
print("  details:", ext["noArgs"]["details"])

assert ext["noArgs"]["isError"] is False
assert ext["noArgs"]["details"] is not None
print()
print("held: the optional parameter accepts a missing argument")
print()
print("The boundary question — an undeclared field — is answered by the chapter's")
print("own test, which asserts that a call with no arguments is valid. An extra")
print("field is a different question; the schema decides, and the schema is the")
print("contract.")

predicted: with base optional, {} is valid and defaults to the working tree

observed above:
  isError: False
  content: Run `git status --short` and report the changed files.
  details: {'base': None, 'diffCmd': 'git status --short'}

held: the optional parameter accepts a missing argument

The boundary question — an undeclared field — is answered by the chapter's
own test, which asserts that a call with no arguments is valid. An extra
field is a different question; the schema decides, and the schema is the
contract.


## Interpretation

An extension is three things, and the chapter's example shows all three:

1. **A tool** — a name, a schema and an `execute()`. Registering it with
   `exposure: "direct"` makes it active immediately.
2. **A command** — `pi.registerCommand` adds a slash command the model can invoke
   or a user can type.
3. **A renderer** — `pi.registerToolRenderer` (1.0.1) draws the tool's call in the
   terminal. Not exercised here.

The practical rules:

* **The schema is the contract.** If the description says optional, the schema must
  say optional. A mismatch is a bug the model will find.
* **Registration is activation.** A `direct` tool is active the moment it is
  registered. The registered-but-not-active gap comes from `deferred` exposure
  (chapter 17) or from an extension that sets the active set itself.
* **A tool returns text, not effects.** The example tool returns an instruction for
  the model to follow. Execution is chapter 17.

## Sources, execution mode and limitations

**Execution mode:** **Run** — the chapter's two extensions loaded unmodified into real sessions, plus the chapter's own test file.

**Evidence scope:** `in_process_runtime`.

### What was read or run

- **Ran** `drivers/ch15-first-extension.ts`: two sessions loading `ch15-first-extension/review-scope.ts` and `ch15-first-extension/hello.ts` unmodified.
- **Ran** `ch15-first-extension/first-extension.test.ts`.
- **Read** `examples/evidence.json`, chapter 15 rows (3 claims).

### Limitations

- **The example tool returns an instruction rather than executing a command.** Execution is chapter 17's subject.
- **`registerToolRenderer` is documented-not-run** by the chapter's suite (`ch17-lim3`); it arrived in 1.0.1.
- **No measurement** of how often a model calls a tool it was offered.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
